# MiWay GTFS Route Efficiency Project - Notebook 3

## Scheduled travel time and speed metrics

This notebook measures how fast MiWay routes are scheduled to operate on the representative weekday: **Tuesday, May 5, 2026**.

It calculates:

1. first and last stop time for every trip,
2. scheduled trip duration,
3. shape-based trip distance,
4. scheduled average speed in km/h,
5. route-level average speed and travel time,
6. slow-route and variable-route flags for later scoring.

This notebook intentionally handles GTFS times beyond midnight, such as `25:10:00`, because GTFS feeds often use hours above 24 for late-night service.

## 1. Import libraries

In [ ]:
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
pd.set_option('display.width', 150)

try:
    import matplotlib.pyplot as plt
    plt.style.use('seaborn-v0_8-whitegrid')
    HAS_MATPLOTLIB = True
except ImportError:
    HAS_MATPLOTLIB = False
    print('matplotlib is not installed, so chart cells will be skipped.')

## 2. Locate project folders and the GTFS ZIP

This notebook can be run from either the project root or the `Notebooks/` folder.

In [ ]:
candidate_roots = [Path.cwd(), Path.cwd().parent]

PROJECT_ROOT = next(
    (root for root in candidate_roots if (root / 'Data' / 'google_transit.zip').exists()),
    None
)

if PROJECT_ROOT is None:
    raise FileNotFoundError('Could not find Data/google_transit.zip from this notebook location.')

DATA_DIR = PROJECT_ROOT / 'Data'
OUTPUT_DIR = PROJECT_ROOT / 'Outputs'
ZIP_PATH = DATA_DIR / 'google_transit.zip'

OUTPUT_DIR.mkdir(exist_ok=True)

print(f'Project root: {PROJECT_ROOT.resolve()}')
print(f'GTFS ZIP:     {ZIP_PATH.resolve()}')
print(f'Outputs:      {OUTPUT_DIR.resolve()}')

## 3. Load GTFS tables

We reload the source feed directly so this notebook can run independently.

In [ ]:
def read_gtfs_table(zip_path: Path, filename: str) -> pd.DataFrame:
    """Read one GTFS text table directly from the ZIP archive."""
    with zipfile.ZipFile(zip_path, 'r') as zf:
        if filename not in zf.namelist():
            raise FileNotFoundError(f'{filename} is missing from {zip_path.name}')
        with zf.open(filename) as file:
            return pd.read_csv(file)


routes = read_gtfs_table(ZIP_PATH, 'routes.txt')
trips = read_gtfs_table(ZIP_PATH, 'trips.txt')
stop_times = read_gtfs_table(ZIP_PATH, 'stop_times.txt')
shapes = read_gtfs_table(ZIP_PATH, 'shapes.txt')
calendar_dates = read_gtfs_table(ZIP_PATH, 'calendar_dates.txt')
feed_info = read_gtfs_table(ZIP_PATH, 'feed_info.txt')

print('Loaded GTFS tables successfully.')

## 4. Select active weekday trips

We use the same representative weekday as Notebooks 1 and 2: **Tuesday, May 5, 2026**.

In [ ]:
ANALYSIS_DATE = pd.Timestamp('2026-05-05')
ANALYSIS_DATE_INT = int(ANALYSIS_DATE.strftime('%Y%m%d'))

feed_start = pd.to_datetime(str(feed_info.loc[0, 'feed_start_date']), format='%Y%m%d')
feed_end = pd.to_datetime(str(feed_info.loc[0, 'feed_end_date']), format='%Y%m%d')

if not (feed_start <= ANALYSIS_DATE <= feed_end):
    raise ValueError('Analysis date is outside the feed validity period.')

active_services = calendar_dates.loc[
    (calendar_dates['date'] == ANALYSIS_DATE_INT) &
    (calendar_dates['exception_type'] == 1),
    'service_id'
].drop_duplicates()

weekday_trips = trips.loc[trips['service_id'].isin(active_services)].copy()

weekday_trips_with_routes = weekday_trips.merge(
    routes,
    on='route_id',
    how='left',
    validate='many_to_one'
)

print(f'Feed period: {feed_start:%B %d, %Y} to {feed_end:%B %d, %Y}')
print(f'Analysis date: {ANALYSIS_DATE:%A, %B %d, %Y}')
print(f'Active service IDs: {len(active_services)}')
print(f'Active trips: {len(weekday_trips):,}')
print(f'Active routes: {weekday_trips["route_id"].nunique()}')

## 5. Parse GTFS times safely

GTFS times can go beyond 24 hours after midnight. Standard clock parsers often fail on these values, so we convert `HH:MM:SS` manually into seconds after the service day starts.

In [ ]:
def gtfs_time_to_seconds(value):
    """Convert a GTFS HH:MM:SS time string to seconds after the service day starts."""
    if pd.isna(value):
        return np.nan
    hours, minutes, seconds = str(value).split(':')
    return int(hours) * 3600 + int(minutes) * 60 + int(seconds)


time_examples = ['05:30:00', '23:59:00', '24:15:00', '27:44:00']
pd.DataFrame({
    'gtfs_time': time_examples,
    'seconds_after_service_day_start': [gtfs_time_to_seconds(t) for t in time_examples],
    'hours_after_service_day_start': [gtfs_time_to_seconds(t) / 3600 for t in time_examples],
})

## 6. Build trip-level travel time table

For each active trip, we use:

- departure time at the first stop,
- arrival time at the last stop,
- number of scheduled stops,
- and total scheduled duration.

In [ ]:
active_trip_ids = weekday_trips_with_routes['trip_id'].drop_duplicates()

weekday_stop_times = stop_times.loc[stop_times['trip_id'].isin(active_trip_ids)].copy()
weekday_stop_times['arrival_seconds'] = weekday_stop_times['arrival_time'].apply(gtfs_time_to_seconds)
weekday_stop_times['departure_seconds'] = weekday_stop_times['departure_time'].apply(gtfs_time_to_seconds)

weekday_stop_times = weekday_stop_times.sort_values(['trip_id', 'stop_sequence'])

trip_time_summary = (
    weekday_stop_times
    .groupby('trip_id', as_index=False)
    .agg(
        first_stop_sequence=('stop_sequence', 'first'),
        last_stop_sequence=('stop_sequence', 'last'),
        first_departure_seconds=('departure_seconds', 'first'),
        last_arrival_seconds=('arrival_seconds', 'last'),
        first_departure_time=('departure_time', 'first'),
        last_arrival_time=('arrival_time', 'last'),
        stop_count=('stop_id', 'count'),
        unique_stop_count=('stop_id', 'nunique')
    )
)

trip_time_summary['scheduled_duration_min'] = (
    trip_time_summary['last_arrival_seconds'] - trip_time_summary['first_departure_seconds']
) / 60

trip_time_summary.head()

## 7. Calculate shape distances

`stop_times.txt` has a `shape_dist_traveled` column, but in this feed it is blank. We therefore calculate trip distance from the shape geometry, using the same Haversine method from Notebook 2.

In [ ]:
def haversine_km(lat1, lon1, lat2, lon2):
    """Vectorized Haversine distance in kilometres."""
    earth_radius_km = 6371.0088

    lat1 = np.radians(lat1)
    lon1 = np.radians(lon1)
    lat2 = np.radians(lat2)
    lon2 = np.radians(lon2)

    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2) ** 2
    c = 2 * np.arcsin(np.sqrt(a))
    return earth_radius_km * c


shapes_sorted = shapes.sort_values(['shape_id', 'shape_pt_sequence']).copy()
shapes_sorted['next_lat'] = shapes_sorted.groupby('shape_id')['shape_pt_lat'].shift(-1)
shapes_sorted['next_lon'] = shapes_sorted.groupby('shape_id')['shape_pt_lon'].shift(-1)

shapes_sorted['segment_km'] = haversine_km(
    shapes_sorted['shape_pt_lat'],
    shapes_sorted['shape_pt_lon'],
    shapes_sorted['next_lat'],
    shapes_sorted['next_lon']
).fillna(0)

shape_lengths = (
    shapes_sorted
    .groupby('shape_id', as_index=False)
    .agg(shape_path_km=('segment_km', 'sum'))
)

shape_lengths.head()

## 8. Combine trips, durations, route names, and distances

This is the main trip-level speed table. Each row is one scheduled bus trip on the analysis date.

In [ ]:
weekday_trip_speed = (
    weekday_trips_with_routes
    .merge(trip_time_summary, on='trip_id', how='left', validate='one_to_one')
    .merge(shape_lengths, on='shape_id', how='left', validate='many_to_one')
)

weekday_trip_speed['scheduled_duration_hr'] = weekday_trip_speed['scheduled_duration_min'] / 60
weekday_trip_speed['scheduled_speed_kmh'] = (
    weekday_trip_speed['shape_path_km'] / weekday_trip_speed['scheduled_duration_hr']
)

weekday_trip_speed['start_hour'] = weekday_trip_speed['first_departure_seconds'] / 3600
weekday_trip_speed['time_period'] = np.select(
    [
        weekday_trip_speed['start_hour'].between(6, 9, inclusive='left'),
        weekday_trip_speed['start_hour'].between(15, 19, inclusive='left'),
        weekday_trip_speed['start_hour'].between(22, 30, inclusive='left'),
    ],
    ['AM peak', 'PM peak', 'late evening'],
    default='midday/other'
)

weekday_trip_speed = weekday_trip_speed.sort_values(['route_short_name', 'direction_id', 'first_departure_seconds'])

weekday_trip_speed[[
    'route_short_name', 'route_long_name', 'trip_id', 'direction_id',
    'first_departure_time', 'last_arrival_time', 'scheduled_duration_min',
    'shape_path_km', 'scheduled_speed_kmh', 'time_period'
]].head(20)

## 9. Filter implausible records for route-level speed metrics

We keep a diagnostic table of unusual records, then exclude trips with impossible or extremely implausible values from route averages. This avoids one bad record distorting the rankings.

In [ ]:
weekday_trip_speed['speed_quality_flag'] = np.select(
    [
        weekday_trip_speed['scheduled_duration_min'].isna(),
        weekday_trip_speed['shape_path_km'].isna(),
        weekday_trip_speed['scheduled_duration_min'] <= 0,
        weekday_trip_speed['shape_path_km'] <= 0,
        weekday_trip_speed['scheduled_speed_kmh'] < 3,
        weekday_trip_speed['scheduled_speed_kmh'] > 80,
    ],
    [
        'missing duration',
        'missing shape distance',
        'non-positive duration',
        'non-positive distance',
        'very slow scheduled speed',
        'very fast scheduled speed',
    ],
    default='ok'
)

speed_diagnostics = weekday_trip_speed.loc[
    weekday_trip_speed['speed_quality_flag'] != 'ok',
    ['route_short_name', 'route_long_name', 'trip_id', 'first_departure_time',
     'last_arrival_time', 'scheduled_duration_min', 'shape_path_km',
     'scheduled_speed_kmh', 'speed_quality_flag']
].copy()

clean_trip_speed = weekday_trip_speed.loc[weekday_trip_speed['speed_quality_flag'] == 'ok'].copy()

print(f'Total active trips: {len(weekday_trip_speed):,}')
print(f'Clean trips used for route averages: {len(clean_trip_speed):,}')
print(f'Trips flagged for review: {len(speed_diagnostics):,}')

speed_diagnostics.head(20)

## 10. Create route-level scheduled speed summary

This table is the main output of Notebook 3. It summarizes typical scheduled speed and travel time by route.

In [ ]:
route_speed_summary = (
    clean_trip_speed
    .groupby(['route_id', 'route_short_name', 'route_long_name'], as_index=False)
    .agg(
        scheduled_trips=('trip_id', 'nunique'),
        directions=('direction_id', 'nunique'),
        avg_trip_distance_km=('shape_path_km', 'mean'),
        median_trip_distance_km=('shape_path_km', 'median'),
        avg_duration_min=('scheduled_duration_min', 'mean'),
        median_duration_min=('scheduled_duration_min', 'median'),
        p10_duration_min=('scheduled_duration_min', lambda s: s.quantile(0.10)),
        p90_duration_min=('scheduled_duration_min', lambda s: s.quantile(0.90)),
        avg_scheduled_speed_kmh=('scheduled_speed_kmh', 'mean'),
        median_scheduled_speed_kmh=('scheduled_speed_kmh', 'median'),
        p10_scheduled_speed_kmh=('scheduled_speed_kmh', lambda s: s.quantile(0.10)),
        p90_scheduled_speed_kmh=('scheduled_speed_kmh', lambda s: s.quantile(0.90)),
        min_scheduled_speed_kmh=('scheduled_speed_kmh', 'min'),
        max_scheduled_speed_kmh=('scheduled_speed_kmh', 'max'),
        avg_stop_count=('stop_count', 'mean')
    )
)

route_speed_summary['duration_spread_min'] = (
    route_speed_summary['p90_duration_min'] - route_speed_summary['p10_duration_min']
)
route_speed_summary['speed_spread_kmh'] = (
    route_speed_summary['p90_scheduled_speed_kmh'] - route_speed_summary['p10_scheduled_speed_kmh']
)

slow_trip_share = (
    clean_trip_speed
    .assign(is_slow_trip=lambda df: df['scheduled_speed_kmh'] < 15)
    .groupby('route_id', as_index=False)
    .agg(share_trips_under_15_kmh=('is_slow_trip', 'mean'))
)

route_speed_summary = route_speed_summary.merge(
    slow_trip_share,
    on='route_id',
    how='left',
    validate='one_to_one'
)

round_cols = [
    'avg_trip_distance_km', 'median_trip_distance_km', 'avg_duration_min',
    'median_duration_min', 'p10_duration_min', 'p90_duration_min',
    'avg_scheduled_speed_kmh', 'median_scheduled_speed_kmh',
    'p10_scheduled_speed_kmh', 'p90_scheduled_speed_kmh',
    'min_scheduled_speed_kmh', 'max_scheduled_speed_kmh', 'avg_stop_count',
    'duration_spread_min', 'speed_spread_kmh', 'share_trips_under_15_kmh'
]
route_speed_summary[round_cols] = route_speed_summary[round_cols].round(2)

route_speed_summary = route_speed_summary.sort_values(
    ['avg_scheduled_speed_kmh', 'scheduled_trips'],
    ascending=[True, False]
)

route_speed_summary.head(25)

## 11. Compare speeds by time period

This quick comparison helps show whether scheduled speeds are lower during peak periods.

In [ ]:
period_speed_summary = (
    clean_trip_speed
    .groupby('time_period', as_index=False)
    .agg(
        trips=('trip_id', 'nunique'),
        avg_speed_kmh=('scheduled_speed_kmh', 'mean'),
        median_speed_kmh=('scheduled_speed_kmh', 'median'),
        avg_duration_min=('scheduled_duration_min', 'mean')
    )
)

period_speed_summary[['avg_speed_kmh', 'median_speed_kmh', 'avg_duration_min']] = (
    period_speed_summary[['avg_speed_kmh', 'median_speed_kmh', 'avg_duration_min']].round(2)
)

period_speed_summary.sort_values('avg_speed_kmh')

## 12. Slowest scheduled routes

These are the routes with the lowest average scheduled speed. In later notebooks, this can be combined with directness and frequency to identify priority routes for advocacy.

In [ ]:
slowest_routes = route_speed_summary[[
    'route_short_name', 'route_long_name', 'scheduled_trips',
    'avg_trip_distance_km', 'avg_duration_min', 'avg_scheduled_speed_kmh',
    'median_scheduled_speed_kmh', 'share_trips_under_15_kmh'
]].head(15)

slowest_routes

## 13. Routes with the most travel-time variation

A large p90-p10 duration spread may indicate routes whose scheduled travel time changes substantially across the day.

In [ ]:
most_variable_routes = route_speed_summary.sort_values(
    'duration_spread_min',
    ascending=False
)[[
    'route_short_name', 'route_long_name', 'scheduled_trips',
    'avg_duration_min', 'p10_duration_min', 'p90_duration_min',
    'duration_spread_min', 'avg_scheduled_speed_kmh'
]].head(15)

most_variable_routes

## 14. Visualize the results

These simple charts are meant for exploration. Later, we can make presentation-ready visuals once the scoring framework is finalized.

In [ ]:
if HAS_MATPLOTLIB:
    fig, ax = plt.subplots(figsize=(10, 6))

    plot_data = slowest_routes.sort_values('avg_scheduled_speed_kmh', ascending=True)
    ax.barh(
        plot_data['route_short_name'].astype(str) + ' - ' + plot_data['route_long_name'].astype(str),
        plot_data['avg_scheduled_speed_kmh'],
        color='#1f77b4'
    )
    ax.set_title('Slowest scheduled MiWay routes on Tuesday, May 5, 2026')
    ax.set_xlabel('Average scheduled speed (km/h)')
    ax.set_ylabel('Route')
    plt.tight_layout()
else:
    print('Install matplotlib to render this chart.')

In [ ]:
if HAS_MATPLOTLIB:
    fig, ax = plt.subplots(figsize=(9, 5))

    ax.hist(clean_trip_speed['scheduled_speed_kmh'], bins=35, color='#2a9d8f', edgecolor='white')
    ax.axvline(clean_trip_speed['scheduled_speed_kmh'].median(), color='#264653', linestyle='--', label='Median trip speed')
    ax.set_title('Distribution of scheduled trip speeds')
    ax.set_xlabel('Scheduled speed (km/h)')
    ax.set_ylabel('Number of trips')
    ax.legend()
    plt.tight_layout()
else:
    print('Install matplotlib to render this chart.')

## 15. Sanity checks

Before exporting, confirm there are no missing route-level metrics and that flagged records are documented.

In [ ]:
sanity_checks = {
    'active weekday trips': len(weekday_trip_speed),
    'clean trips used for speed metrics': len(clean_trip_speed),
    'flagged trips excluded from route averages': len(speed_diagnostics),
    'active routes': weekday_trips_with_routes['route_id'].nunique(),
    'routes in speed summary': len(route_speed_summary),
    'routes missing avg speed': route_speed_summary['avg_scheduled_speed_kmh'].isna().sum(),
    'minimum route avg speed km/h': route_speed_summary['avg_scheduled_speed_kmh'].min(),
    'maximum route avg speed km/h': route_speed_summary['avg_scheduled_speed_kmh'].max(),
}

pd.Series(sanity_checks, name='value').to_frame()

## 16. Export Notebook 3 outputs

The route-level output will feed into the final efficiency scoring notebook.

In [ ]:
trip_speed_path = OUTPUT_DIR / 'weekday_trip_speed_metrics_2026-05-05.csv'
route_speed_path = OUTPUT_DIR / 'route_scheduled_speed_metrics_2026-05-05.csv'
period_speed_path = OUTPUT_DIR / 'period_scheduled_speed_summary_2026-05-05.csv'
diagnostics_path = OUTPUT_DIR / 'trip_speed_diagnostics_2026-05-05.csv'

weekday_trip_speed.to_csv(trip_speed_path, index=False)
route_speed_summary.to_csv(route_speed_path, index=False)
period_speed_summary.to_csv(period_speed_path, index=False)
speed_diagnostics.to_csv(diagnostics_path, index=False)

print('Exported:')
print('-', trip_speed_path)
print('-', route_speed_path)
print('-', period_speed_path)
print('-', diagnostics_path)

# What this notebook accomplished

Notebook 3 created scheduled travel time and speed metrics for the representative weekday.

The main route-level output includes:

- average trip distance,
- average and median scheduled duration,
- average and median scheduled speed,
- p10/p90 duration and speed spread,
- share of trips scheduled under 15 km/h,
- and diagnostics for unusual records.

## Next notebook

Notebook 4 should calculate **frequency and service availability**:

1. first and last trip by route,
2. service span,
3. trips per direction,
4. average headway,
5. peak and off-peak headways,
6. directional balance.